In [3]:
#Import neccessary libraries
from scipy.stats import qmc
import numpy as np
import pandas as pd

from scipy.stats import norm
from scipy.optimize import minimize
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.gaussian_process.kernels import Matern

In [4]:
#Function 1
#Extracting updated data and turning it into a pandas dataframe
data = pd.read_csv('Data/Week 6 - Function 1.csv') 
columns = ['Input 1', 'Input 2', 'Outputs']
#Remove columns with NaN values
data = data.dropna(axis = 1)
#Manually inputting outputs because of issues with reading small values
outputs = pd.DataFrame(np.array([1.322677E-79, 1.033078E-46, 7.710875E-16,3.341771E-124,-0.003606063
                   ,-2.159249E-54, -2.089093E-91,2.535001E-40,3.606771E-81, 6.229856E-48,
                   -7.2631434844889E-45, 2.42005112861639E-115, 0, 
                                -5.71190370994002E-268,2.67528799107424E-09]))
data = data.join(outputs)
data = data.drop(columns = 'Unnamed: 2')
data.columns = columns
data

,Input 1,Input 2,Outputs
0,0.319404,0.762959,1.322677e-79
1,0.574329,0.879898,1.033078e-46
2,0.731024,0.733000,7.710875e-16
3,0.840353,0.264732,3.341771e-124
4,0.650114,0.681526,-3.606063e-03
5,0.410437,0.147554,-2.159249e-54
6,0.312691,0.078723,-2.089093e-91
7,0.683418,0.861057,2.535001e-40
8,0.082507,0.403488,3.606771e-81
9,0.883890,0.582254,6.229856e-48


In [1]:
#Extract the Data Into Numpy Arrays
X = np.array(data[['Input 1', 'Input 2']])
Y = np.array(data[['Outputs']])


#Visualise already explored points
import matplotlib.pyplot as plt
plt.scatter(X[:, 0], X[:,1])
plt.xlabel('Input 1')
plt.ylabel('Input 2')
plt.show()

NameError: name 'np' is not defined

In [6]:
# Find the pearson correlations matrix
data.corr(method = 'pearson')

,Input 1,Input 2,Outputs
Input 1,1.000000,0.007319,-0.117541
Input 2,0.007319,1.000000,-0.136670
Outputs,-0.117541,-0.136670,1.000000


In [1]:
#Plotting each Input against log of the output
#Plotting the log of the (absolute value) output since the values are very small
fig, ax = plt.subplots(1,2, figsize = (10,8))
eps = 1e-300
ax[0].scatter(X[:,0], np.log(np.abs(Y)+eps))
ax[1].scatter(X[:,1], np.log(np.abs(Y)+eps))

#Creating titles and axis labels
ax[0].set_title('Input 1 against log Output')
ax[1].set_title('Input 2 against log Output')

ax[0].set_xlabel('Input 1')
ax[0].set_ylabel('Output')

ax[1].set_xlabel('Input 2')
ax[1].set_ylabel('Output')
plt.show()

NameError: name 'plt' is not defined

In [8]:
#We now set the Kernel to be the Matern Kernel
nu = 2.5
kernel = Matern(length_scale=1.0, length_scale_bounds= (1e-2,1e1), nu = nu)
gp_best = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y = True)
gp_best.fit(X,Y)
#Use latin hypercube sampling to create a set of candidate points 
d = 2
n = 100000
sampler = qmc.LatinHypercube(d, seed = 42)
grid = sampler.random(n)   # samples in [0,1]^d

#Fit the GP regressor to the candidate points
mu, std = gp_best.predict(grid, return_std = True)

/opt/conda/envs/anaconda-ai-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [9]:
#Calculating expected improvement
best_y = np.max(Y)
eta = 0.01
z = (mu - best_y - eta) / (std + 1e-12)
ei = (mu - best_y - eta ) * norm.cdf(z) + std * norm.pdf(z)
#Find Candidate value that maximises expected improvement 
best_idx = np.argmax(ei)
x_next = grid[best_idx]
np.round(x_next,6)

array([0.335936, 0.099268])

In [15]:
#Calculate UCB 
beta = 2.5 #set exploration parameter
UCB = mu + beta * std
#Find Candidate value the maximises UCB
best_idx = np.argmax(UCB)
x_next = grid[best_idx]
np.round(x_next,6)

array([0.329743, 0.09222 ])